# 05 规则引擎模块 (core.rules)

覆盖 Rule 对象、规则挖掘、表达式优化、规则集评估报告。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from hscredit import (germancredit, init_setting, Rule,
    get_columns_from_query, optimize_expr, beautify_expr, get_expr_variables,
    ruleset_report,
)
from hscredit.core.rules.mining import SingleFeatureRuleMiner, MultiFeatureRuleMiner
init_setting()
df = germancredit()
target = 'creditability'
print(df.shape)

## 1. Rule 对象 —— 创建与评估

In [ ]:
# 创建单条规则
r1 = Rule('`duration.in.month` > 24 and `credit.amount` > 3000')
print('规则:', r1.expr)
print('涉及变量:', r1.get_variables())

# 应用规则到数据
hit = r1.apply(df)
print(f'命中率: {hit.mean():.2%}')
print(f'命中坏率: {df.loc[hit, target].mean():.2%}')
print(f'未命中坏率: {df.loc[~hit, target].mean():.2%}')

## 2. 规则集评估报告

In [ ]:
rules = [
    '`duration.in.month` > 36',
    '`credit.amount` > 5000',
    '`age.in.years` < 25',
    '`duration.in.month` > 24 and `credit.amount` > 3000',
]
report = ruleset_report(df, rules=rules, target=target)
print(report.columns.tolist())
report.head()

## 3. 表达式工具

In [ ]:
expr = '(`credit.amount` > 3000) AND (`duration.in.month` <= 36 OR `age.in.years` < 30)'
print('原始:', expr)
print('优化:', optimize_expr(expr))
print('美化:', beautify_expr(expr))
print('变量:', get_expr_variables(expr))
print('列名:', get_columns_from_query(expr, df))

## 4. 规则挖掘 —— 单特征

In [ ]:
num_feats = df.select_dtypes('number').columns.drop(target).tolist()[:5]
miner = SingleFeatureRuleMiner(
    target=target,
    min_lift=1.5,
    min_support=0.05,
    max_n_bins=5,
)
miner.fit(df, features=num_feats)
results = miner.rules_df_
print(f'挖掘到 {len(results)} 条规则')
results.head(10)

## 5. 规则挖掘 —— 多特征组合

In [ ]:
multi_miner = MultiFeatureRuleMiner(
    target=target,
    min_lift=2.0,
    min_support=0.03,
    max_depth=2,
    max_n_bins=4,
)
multi_miner.fit(df, features=num_feats[:4])
print(f'多特征挖掘规则数: {len(multi_miner.rules_df_)}')
multi_miner.rules_df_.head(10)